In [366]:
from microscope_calibration.common.model import Parameters4DSTEM, DescanError, trace, PixelYX, Model4DSTEM, Ray

from temgym_core.run import run_iter

import jax
import jax.numpy as jnp
import numpy as np
import sympy as sym
from sympy import S

In [3]:
trace(
    params=p,
    scan_pos=PixelYX(0., 0.),
    source_dy=0.1,
    source_dx=0.1,
)

OrderedDict([('source',
              ResultSection(component=PointSource(z=0, semi_conv=0.001, offset_xy=CoordsXY(x=0.0, y=0.0)), ray=Ray(x=0.0, y=0.0, dx=0.1, dy=0.1, z=0, pathlength=0.0, _one=1.0), sampling=None)),
             ('overfocus',
              ResultSection(component=Propagator(distance=0.01, propagator=<temgym_core.propagator.FreeSpaceParaxial object at 0x7f0e1bb47b60>), ray=Ray(x=0.001, y=0.001, dx=0.1, dy=0.1, z=0.01, pathlength=0.01, _one=1.0), sampling=None)),
             ('scanner',
              ResultSection(component=Scanner(z=0.01, scan_pos_x=np.float64(-1.989900167499164e-07), scan_pos_y=np.float64(-1.0199496670849987e-07), scan_tilt_x=0.0, scan_tilt_y=0.0), ray=Ray(x=np.float64(0.0009998010099832502), y=np.float64(0.0009998980050332914), dx=0.1, dy=0.1, z=0.01, pathlength=0.01, _one=1.0), sampling=None)),
             ('specimen',
              ResultSection(component=Plane(z=0.01), ray=Ray(x=np.float64(0.0009998010099832502), y=np.float64(0.0009998980050332

In [4]:
def test(source_dx):
    tr = trace(
        params=p,
        scan_pos=PixelYX(0.1, 0.1),
        source_dy=0.23,
        source_dx=source_dx,
    )
    return tr['detector'].sampling['detector_px'].x

In [5]:
test(0.1)

np.float64(2020.1000000000001)

In [6]:
def testjax(source_dx):
    tr = trace(
        params=p.derive(xp=jnp),
        scan_pos=PixelYX(0., 0.),
        source_dy=0.,
        source_dx=source_dx,
    )
    return tr['detector'].sampling['detector_px'].x

In [7]:
jax.grad(testjax)(0.)

Array(20200., dtype=float64, weak_type=True)

In [8]:
x = sym.Symbol('x')
expr = x*20000
result = expr.subs(x,5) #x=5
result

100000

In [9]:
test(np.array((x)))

20200.0*x + 0.1

In [10]:
x * 1 == x - x + x
#x = sym.Symbol('x', zero=True)

True

In [11]:
x = sym.Symbol('x')
model = Model4DSTEM.build(params=p, scan_pos=PixelYX(0.01, 0.2))
ray = model.make_source_ray(source_dx=x, source_dy=0.2).ray

res = list(run_iter(ray=ray, components=model.components))

comp, r = res.pop(0)
print(ray.dx)

x


In [12]:
# does not reconize e.g. np.array((x, 1)) as sympy
def is_sympy(x) -> bool:
    return isinstance(x, sym.Expr)

def sympy_equals(a, b) -> bool:
    return sym.simplify(a - b).is_zero is True

In [13]:
def equals(ray1: Ray, ray2: Ray) -> bool:
    for key in ray1.__dict__.keys():
        if (is_sympy(ray1.__dict__[key]) or is_sympy(ray2.__dict__[key])):
            if not sympy_equals(ray1.__dict__[key], ray2.__dict__[key]):
                return False
        else:
            if ray1.__dict__[key] != ray2.__dict__[key]:
                return False
    return True

In [14]:
equals(Ray(x, 0, 0, 0, 0, 0), Ray(np.array((x)), 0, 0, 0, 0, 0)) is True

True

In [15]:
sym.simplify(np.array((x)) - x)

0

In [16]:
sym.simplify(np.array((x,)) - x)

[0]

In [17]:
sym.simplify(np.array((x,)) - x) == sym.Array([0])

True

In [18]:
isinstance(sym.simplify(np.array((x, )) - x), sym.Array)

True

In [19]:
#_protected(...)
#__private(...)

In [20]:
equals(Ray(S(0), x, 0, 0, 0, 0), Ray(0.0, 2*x, 0, 0, 0, 0)) is True

False

In [21]:
is_sympy(list((1,2,x)))

False

In [22]:
is_sympy(Ray(x, 0, 0, 0, 0, 0))

False

In [23]:
l = np.array((x)) # ATTENTION: it works for this
sym.simplify(l-l == 0) is sym.logic.boolalg.BooleanTrue()
#l-l

True

In [24]:
def test_eq(source):
    tr = trace(r)
    return tr['detector'].sampling['detector_px'].x

In [25]:
test(x)

20200.0*x + 0.1

In [26]:
x*0.5 +1 == x*2 - 1.5*x + 2 - 1.0

False

In [27]:
x*0.5 + 1 - (x*2 - 1.5*x + 2 - 1.0)==0

True

In [28]:
sym.simplify(x*0.5 +1 - (x*2 - 1.5*x + 2 - 1.0))==0

True

In [29]:
a, b = sym.symbols('a b')
(a+b)**2 - (a**2 +2*a*b + b**2) == 0

False

In [30]:
sym.simplify((a+b)**2 - (a**2 +2*a*b + b**2)) == 0

True

In [31]:
sym.simplify((a+b)**2 - (a**2 +2*a*b + b**2) == 0) is sym.logic.boolalg.BooleanTrue() # comparison !!!ATTENTION it does not work

False

In [32]:
 sym.logic.boolalg.BooleanTrue()

True

In [33]:
0-0.0 == 0

True

In [34]:
a = np.array((23., 42.))

In [35]:
sym.simplify(a - a*1 == 0) is sym.logic.boolalg.BooleanTrue()

False

In [36]:
isinstance(x, sym.Expr) is True # proving if is sympy

True

In [37]:
np.array((x, x)) == x

array([ True,  True])

In [38]:
b = jax.numpy.array((23., 42.))

In [39]:
#bool(sym.simplify(b - b == 0))

In [40]:
x

x

In [41]:
x.__class__.__name__

'Symbol'

In [42]:
(S(0 - 0.0) == 0) is True

False

In [43]:
(0 - 0.0 == 0) is True

True

In [44]:
S(0 - 0.0 == 0) is S.true

True

In [45]:
S(0-0.0) is S.Zero

False

In [46]:
sym.simplify(0-0.0).is_zero is True

True

In [47]:
a, b = sym.symbols('a b')
sym.simplify((a+b)**2 - (a**2 +2*a*b + b**2)).is_zero is True

True

In [48]:
sym.simplify(1*x-1.0*x - 1.0 + 1).is_zero is True

True

In [49]:
sym.simplify(0.0).is_zero is True

True

In [50]:
S(0.0).is_zero

True

In [51]:
S(0).is_zero

True

In [52]:
is_sympy(S(0))

True

In [53]:
sym.Or(is_sympy(np.array((x, 1))), is_sympy(ray.__dict__['x'])) is S.true

False

In [54]:
is_sympy(r.__dict__['x'])

True

In [55]:
r.__dict__['x']

0

In [56]:
(is_sympy(np.array((x, 1))) or is_sympy(ray.__dict__['x'])) is True

False

In [57]:
isinstance(np.array((x, 1)), sym.Expr)

False

In [58]:
type(np.array((x, 1)))

numpy.ndarray

In [59]:
a, b = sym.symbols('a b')
sym.simplify((a+b)**2 - (a**2 +2*a*b + b**2)).is_zero is True

True

In [60]:
l = [x]
l.append(l)
x = sym.Symbol('x')

In [61]:
l

[x, [...]]

In [62]:
from typing import Iterable #Sequence # typing or collections.abc?

In [63]:
def is_sympy_recursive(a, checked=set()):
    if id(a) in checked:
        return False
    checked.add(id(a))
    if isinstance(a, Iterable):
        return any(is_sympy_recursive(x, checked=checked) for x in a)
    else:
        return isinstance(a, sym.Expr)

In [64]:
is_sympy_recursive(l, checked=set())

True

In [65]:
type(sym.simplify(0-0.0)) == 0.0

False

In [66]:
is_sympy(S(6))

True

In [67]:
f = sym.Function('f')(x)
is_sympy(f.subs(x, 1))

True

In [68]:
A = sym.Matrix([[1,2],[2,3]])

In [69]:
is_sympy_recursive(A)

False

In [70]:
def is_sympy_basic(x) -> bool:
    return isinstance(x, sym.Basic)

In [81]:
is_sympy_basic(sym.sympify(sym.Matrix([1])))

True

In [87]:
sym.sympify([1,2])

[1, 2]

In [72]:
is_sympy(x> 0)

False

In [73]:
is_sympy_basic(S(A))

True

In [74]:
Ar = sym.Array([1,2])
B = sym.Matrix([[1,2],[3,4]])
isinstance(Ar, sym.NDimArray)
sym.simplify(A-B)
isinstance(x, (sym.Basic, sym.MatrixBase, sym.NDimArray))

True

In [75]:
((a+b)**2).equals(a**2 +2*a*b+b**2)

True

In [76]:
def equals_with_sympy_equals(ray1: Ray, ray2: Ray) -> bool:
    for key in ray1.__dict__.keys():
        if (is_sympy(ray1.__dict__[key]) or is_sympy(ray2.__dict__[key])):
            if not ray1.__dict__[key].equals(ray2.__dict__[key]):
                return False
        else:
            if ray1.__dict__[key] != ray2.__dict__[key]:
                return False
    return True

In [77]:
equals_with_sympy_equals(Ray((a+b)**2, 0, 0, 0, 0, 0), Ray(a**2 + 2*a*b + b**2, 0, 0, 0, 0, 0))

True

In [78]:
equals_with_sympy_equals(Ray(0.0,0,0,0,0,0), Ray(S(0),0,0,0,0,0))

AttributeError: 'float' object has no attribute 'equals'

In [ ]:
equality(Ray(0.0,0,0,0,0,0), Ray(S(0),0,0,0,0,0))

In [ ]:
S(0).equals(0.0)

In [ ]:
sym.Number(0.0).equals(S(0))

In [ ]:
sym.simplify(0.0).equals(S(0))

In [ ]:
sym.Expr(0.0)

In [ ]:
from microscope_calibration.common.model import rotate

In [ ]:
import numpy.typing as npt
import jax.numpy as jnp

In [ ]:
isinstance(jnp.array((1, 2, 3)), npt.ArrayLike)

In [ ]:
jnp.ndarray()

In [ ]:
type(jnp.ndarray())

In [ ]:
def npt_test(a: npt.ArrayLike):
    return 'it works'

In [ ]:
f = jnp.ndarray()
npt_test(f)

In [ ]:
jnp.ndarray()

In [ ]:
jnp.array((1, 2, 3))

In [ ]:
isinstance(23, (float, str))

In [ ]:
from typing import Union, Optional

Optional[Union[int, float]]

In [ ]:
int | float | None

In [369]:
from sympy.abc import a, b, c, d, e, f, g, h, i, j, k, l, m, n, o, p, q, r, s, t, u, v, w
par = Parameters4DSTEM(
    overfocus=0.01*a,
    scan_pixel_pitch=1e-6*b,
    scan_center=PixelYX(0.1*c, 0.2*d),
    scan_rotation=0.01*e,
    camera_length=1.0*f,
    detector_pixel_pitch=50e-6*g,
    detector_center=PixelYX(0.1*h, 0.1*i),
    semiconv=1e-3*j,  # radian
    flip_factor=1.0*k,
    descan_error=DescanError(pxo_pxi=l, pxo_pyi=m, pyo_pxi=n, pyo_pyi=o, sxo_pxi=p, sxo_pyi=q, syo_pxi=r, syo_pyi=s, offpxi=t, offpyi=u, offsxi=v, offsyi=w),
    xp=np
)

In [160]:
par.descan_error

DescanError(pxo_pxi=l, pxo_pyi=m, pyo_pxi=n, pyo_pyi=o, sxo_pxi=p, sxo_pyi=q, syo_pxi=r, syo_pyi=s, offpxi=t, offpyi=u, offsxi=v, offsyi=w)

In [161]:
def test_adjust():
    answer = Parameters4DSTEM.adjust_scan_pixel_pitch(
        par,
        scan_pixel_pitch = 2.5
    )
    return answer.descan_error

In [162]:
test_adjust()

DescanError(pxo_pxi=4.0e-7*b*l, pxo_pyi=4.0e-7*b*m, pyo_pxi=4.0e-7*b*n, pyo_pyi=4.0e-7*b*o, sxo_pxi=4.0e-7*b*p, sxo_pyi=4.0e-7*b*q, syo_pxi=4.0e-7*b*r, syo_pyi=4.0e-7*b*s, offpxi=t, offpyi=u, offsxi=v, offsyi=w)

In [172]:
par.descan_error.__getitem__(0)

l

In [183]:
for item in par.descan_error._fields:
    print(item)

pxo_pxi
pxo_pyi
pyo_pxi
pyo_pyi
sxo_pxi
sxo_pyi
syo_pxi
syo_pyi
offpxi
offpyi
offsxi
offsyi


In [221]:
test_adjust().pxo_pxi / par.descan_error.pxo_pxi

4.0e-7*b

In [236]:
add, div = sym.symbols('add div')
sym.solve([(test_adjust().pxo_pxi - add) * div / par.scan_pixel_pitch - par.descan_error.pxo_pxi, (test_adjust().pxo_pyi - add) * div / par.scan_pixel_pitch - par.descan_error.pxo_pyi], [add, div], dict=True)

[{add: 0.0, div: 2.50000000000000}]

In [237]:
res = sym.symbols('res')
def adjust_new(param, scan_pixel_pitch):
    return sym.solve(param.descan_error.pxo_pxi * param.scan_pixel_pitch / scan_pixel_pitch - res, res)

In [239]:
adjust_new(par, scan_pixel_pitch=2.5)

[4.0e-7*b*l]

In [389]:
res = sym.symbols('res')
def adjust_multiple(param, scan_pixel_pitch):
    result_list = []
    for key in param.descan_error._asdict().keys():
        result_list.append(sym.solve(param.descan_error._asdict()[key] * param.scan_pixel_pitch / scan_pixel_pitch - res, res))
    return result_list

In [390]:
adjust_multiple(par, 2.5)

[[4.0e-7*b*l],
 [4.0e-7*b*m],
 [4.0e-7*b*n],
 [4.0e-7*b*o],
 [4.0e-7*b*p],
 [4.0e-7*b*q],
 [4.0e-7*b*r],
 [4.0e-7*b*s],
 [4.0e-7*b*t],
 [4.0e-7*b*u],
 [4.0e-7*b*v],
 [4.0e-7*b*w]]

In [263]:
par.descan_error.__getattribute__('pxo_pxi')

l

In [388]:
par.descan_error._asdict().keys()

dict_keys(['pxo_pxi', 'pxo_pyi', 'pyo_pxi', 'pyo_pyi', 'sxo_pxi', 'sxo_pyi', 'syo_pxi', 'syo_pyi', 'offpxi', 'offpyi', 'offsxi', 'offsyi'])